In [30]:
import json
from pathlib import Path

from tqdm.auto import tqdm

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans
from sklearn.decomposition import IncrementalPCA
from sklearn.metrics import silhouette_score


In [47]:
# DATA DIRECTORIES
PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'newyorker_caption_contest'
JSON_DIR = DATA_DIR / 'contests.json'
CSV_DIR = DATA_DIR / 'data'
CSV_CLEAN_DIR = DATA_DIR / 'data_cleaned'
IMG_DIR = DATA_DIR / 'images'
DOC_THEMES_DIR = DATA_DIR / 'THEMES_RESULTS_V1'/'docs_with_themes.csv'
THEMES_TFIDF_DIR=DATA_DIR/'THEMES_RESULTS_V1'/'TF_IDF_themes_scores.csv'
THEMES_DIR=DATA_DIR/'THEMES_RESULTS_V1'/'themes.csv'




## 0. Cleaning the data set with spacy

We used spacy to clean the raw data set to have more meaningfull themes

In [48]:

docs = []

for csv_path in sorted(CSV_CLEAN_DIR.glob("*.csv")):
    df = pd.read_csv(csv_path)
    contest_id = csv_path.stem

    for idx, row in df.iterrows():
        text = str(row.get("caption", "")).strip()
        if not text:
            continue

        docs.append({
            "contest_id": contest_id,
            "caption_id": idx,
            "text": text,

            # metadata (if present)
            "rank": row.get("rank"),
            "mean": row.get("mean"),
            "precision": row.get("precision"),
            "votes": row.get("votes"),
            "funny": row.get("funny"),
            "somewhat_funny": row.get("somewhat_funny"),
            "not_funny": row.get("not_funny"),
        })

docs_df = pd.DataFrame(docs)
print("docs_df shape:", docs_df.shape)


docs_df shape: (2263088, 10)


In [49]:
docs_df.head()

,contest_id,caption_id,text,rank,mean,precision,votes,funny,somewhat_funny,not_funny
0,510_cleaned,0,I'm a congressman--obstruction is my job.,0,1.913043,0.094022,69,18,27,24
1,510_cleaned,1,"I'm what they mean when they say, 'The middle ...",1,1.842105,0.191381,19,5,6,8
2,510_cleaned,2,Does this suit make me look flat?,2,1.711111,0.112915,45,8,16,21
3,510_cleaned,3,"When the right woman comes along, I'll know it.",3,1.625000,0.116657,32,3,14,15
4,510_cleaned,4,"I used to lie in the gutter, but then I quit d...",4,1.617647,0.133610,34,6,9,19


In [51]:
custom_stopwords = {
    "like", "don", "ll", "kinda", "sorta",
}

for w in custom_stopwords:
    nlp.Defaults.stop_words.add(w)

In [ ]:
import spacy
from tqdm import tqdm

# Charger spaCy
nlp = spacy.load("en_core_web_sm")

# Activer tqdm pour pandas
tqdm.pandas()

def clean_caption(text):
    """
    Clean a caption by:
    - running spaCy
    - removing stop words and non-content words
    - keeping only alphabetic NOUN/PROPN/ADJ/VERB
    - lemmatizing and lowercasing
    """
    doc = nlp(str(text))  # str() in case of NaN
    tokens = []
    for token in doc:
        if (
            token.is_alpha
            and not token.is_stop
            and token.pos_ in {"NOUN", "PROPN", "ADJ", "VERB"}
        ):
            tokens.append(token.lemma_.lower())
    return " ".join(tokens)

# Apply with progress bar
docs_df["clean_text"] = docs_df["text"].progress_apply(clean_caption)


 96%|█████████▋| 2178630/2263088 [1:29:29<03:25, 411.21it/s]

In [ ]:
docs_df.head()

In [ ]:
cols_export = [
    "doc_id", "contest_id", "text", "rank", "mean", "precision",
    "votes", "not_funny", "somewhat_funny", "funny"
]

# export csv
docs_df.to_csv("docs_clean_text.csv", columns=cols_export, index=False)

In [50]:
import spacy
nlp = spacy.load("en_core_web_sm")

def clean_caption(text):
    doc = nlp(str(text))
    toks = [
        t.lemma_.lower()
        for t in doc
        if t.is_alpha and not t.is_stop and t.pos_ in {"NOUN","PROPN","ADJ","VERB"}
    ]
    return " ".join(toks)

docs_df["clean_text"] = docs_df["text"].apply(clean_caption)

KeyboardInterrupt: 

In [29]:
# Count captions per theme
theme_freq = (
    docs_df["theme_id"]
    .value_counts()
    .reset_index()
    .rename(columns={"index": "theme_id", "theme_id": "n_captions"})
)

# Show top 10 most frequent themes
theme_freq.head(10)



,n_captions,count
0,75,100287
1,42,97886
2,35,53470
3,8,48931
4,57,47142
5,44,47141
6,39,45188
7,73,44259
8,32,43654
9,13,43395


Check if recurrent words in our top themes are in the library

In [ ]:

words_to_check = ["like", "don't", "don", "just", "i", "ll", "im", "kinda", "sorta"]

for w in words_to_check:
    doc = nlp(w)
    token = doc[0]
    print(f"{w:10} -> is_stop={token.is_stop}, lemma={token.lemma_}")


like       -> is_stop=True, lemma=like
don't      -> is_stop=True, lemma=do
don        -> is_stop=True, lemma=don
just       -> is_stop=True, lemma=just
i          -> is_stop=True, lemma=I
ll         -> is_stop=True, lemma=ll
im         -> is_stop=True, lemma=I
kinda      -> is_stop=True, lemma=kinda
sorta      -> is_stop=True, lemma=sorta


So It seems we have to add a few stop words 


In [8]:
custom_stopwords = {
    "like", "don", "ll", "kinda", "sorta",
}

for w in custom_stopwords:
    nlp.Defaults.stop_words.add(w)


In [ ]:

def clean_caption(text):
    """
    Clean a caption by:
    - running spaCy
    - removing stop words and non-content words
    - keeping only alphabetic NOUN/PROPN/ADJ/VERB
    - lemmatizing and lowercasing
    """
    doc = nlp(str(text))  # str() in case of NaN
    tokens = []
    for token in doc:
        if (
            token.is_alpha
            and not token.is_stop
            and token.pos_ in {"NOUN", "PROPN", "ADJ", "VERB"}
        ):
            tokens.append(token.lemma_.lower())
    return " ".join(tokens)
docs_df["clean_text"] = docs_df["text"].apply(clean_caption)




Now we applied Bert, clustering then TF-IDF on the cleaned text and we obtained the different themes.

## 1. Visualizing theme frequencies

In [45]:
#visualize the theme csv
themes_df = pd.read_csv(THEMES_DIR)
themes_df.head(10)

,theme_id,theme_label,nb_instances,mu_rank,mu_mean,mu_precision,mu_not_funny,mu_somewhat_funny,mu_funny,TF_IDF_terms
0,0,Gluten Free,14451,3845.514545,1.194643,0.055576,85.271608,18.694900,6.303924,"gluten, vegan, gluten free, free, vegetarian, ..."
1,1,Tell Time,25116,2775.081794,1.220181,0.056183,93.574614,26.224399,10.973364,"time, minute, think time, clock, think, tell, ..."
2,2,Casual Friday,28622,2879.948578,1.218212,0.054937,92.117951,26.062120,10.309971,"day, week, today, friday, tomorrow, weekend, d..."
3,3,Puppy,39062,2884.836397,1.212637,0.048746,101.188503,26.841634,10.520574,"dog, cat, pet, bark, leash, animal, puppy, vet..."
4,4,Howl,29277,3313.285684,1.210709,0.048704,98.455903,25.269537,10.266222,"bear, elephant, sheep, animal, tail, giraffe, ..."
5,5,Elephant Room,31108,2891.192361,1.216552,0.056607,88.493066,25.622068,10.950446,"room, elephant, elephant room, table, couch, s..."
6,6,Swear,50342,3107.824964,1.204843,0.051953,93.781137,23.799213,9.660919,"say, honey, mean, hear, know, stop, tell, let,..."
7,7,Insurance Cover,28619,2929.875181,1.224620,inf,88.301478,24.979349,10.376813,"insurance, cover, policy, pay, warranty, plan,..."
8,8,Funny,57199,2852.007847,1.219179,0.054308,93.943968,25.733160,10.436511,"thing, mean, guess, right, bad, guy, funny, po..."
9,9,Perspective,49493,2864.220444,1.218970,0.057265,89.027200,24.250293,9.424928,"think, believe, understand, know, read, mean, ..."


In [ ]:
# Load TF-IDF theme scores
themes_dfidf = pd.read_csv(THEMES_TFIDF_DIR)
themes_dfidf.head(10)

In [ ]:
# Link TF-IDF terms to their theme labels via theme_id
term_theme_df = (
    themes_dfidf[["theme_id", "term"]]
    .merge(themes_df[["theme_id", "theme_label"]], on="theme_id", how="left")
    .loc[:, ["term", "theme_label"]]
)
term_theme_df.head(10)